# 02 — Better evaluation & a content-based model

This notebook adds two things: rank-based metrics (precision@k and recall@k), which matter more than RMSE for recommendations, and a second model — content-based filtering using TF-IDF on course text.

## 1. Imports & load data

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

reviews = pd.read_csv('../data/Coursera_reviews.csv')
courses = pd.read_csv('../data/Coursera_courses.csv')
print('reviews:', reviews.shape, '| courses:', courses.shape)
print('course columns:', list(courses.columns))

reviews: (1454711, 5) | courses: (623, 4)
course columns: ['name', 'institution', 'course_url', 'course_id']


## 2. Rank-based evaluation: precision@k and recall@k

RMSE only tells us how close predicted ratings are. For a recommender, we care
whether the **top k** items we show are actually relevant. We treat a rating of
4 or 5 as 'relevant' and measure how many of our top-k picks are relevant.

In [2]:
# Prepare data (same as notebook 01)
df = reviews[['reviewers', 'course_id', 'rating']].dropna()
df.columns = ['user', 'item', 'rating']
active = df['user'].value_counts()
df = df[df['user'].isin(active[active >= 3].index)]

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user', 'item', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

model = SVD(random_state=42)
model.fit(trainset)
predictions = model.test(testset)
print('made', len(predictions), 'predictions on the test set')

made 356999 predictions on the test set


In [3]:
def precision_recall_at_k(predictions, k=10, threshold=4.0):
    """Standard precision@k and recall@k for recommender systems."""
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions, recalls = {}, {}
    for uid, ratings in user_est_true.items():
        ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum(true_r >= threshold for (_, true_r) in ratings)
        n_rec_k = sum(est >= threshold for (est, _) in ratings[:k])
        n_rel_rec_k = sum((true_r >= threshold and est >= threshold)
                          for (est, true_r) in ratings[:k])
        precisions[uid] = n_rel_rec_k / n_rec_k if n_rec_k else 0
        recalls[uid] = n_rel_rec_k / n_rel if n_rel else 0
    return precisions, recalls

precisions, recalls = precision_recall_at_k(predictions, k=10, threshold=4.0)
print(f'Precision@10: {sum(precisions.values()) / len(precisions):.3f}')
print(f'Recall@10   : {sum(recalls.values()) / len(recalls):.3f}')

Precision@10: 0.952
Recall@10   : 0.949


## 3. Content-based model (TF-IDF)

Instead of using ratings, recommend courses whose **text is similar**. We turn
each course's name (and any skills/description columns) into a TF-IDF vector and
measure cosine similarity. This helps with *new* courses that have no ratings yet.

In [4]:
# Build a text field from the descriptive columns (name, institution, skills, description).
name_col = 'name' if 'name' in courses.columns else courses.columns[0]
text_cols = [c for c in [name_col, 'institution', 'skills', 'description'] if c in courses.columns]
courses['text'] = courses[text_cols].fillna('').astype(str).agg(' '.join, axis=1)

tfidf = TfidfVectorizer(stop_words='english')
matrix = tfidf.fit_transform(courses['text'])
similarity = cosine_similarity(matrix)
print('TF-IDF matrix:', matrix.shape)

TF-IDF matrix: (623, 1277)


In [5]:
index = pd.Series(courses.index, index=courses[name_col]).drop_duplicates()

def similar_courses(course_name, n=10):
    if course_name not in index:
        return f'"{course_name}" not found. Try one of: ' + ', '.join(courses[name_col].head(5))
    idx = index[course_name]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n + 1]
    return courses[name_col].iloc[[i for i, _ in scores]].tolist()

# Try it on the first course in the catalogue
example = courses[name_col].iloc[0]
print('Because you liked:', example)
for c in similar_courses(example, 10):
    print('  -', c)

Because you liked: Machine Learning
  - Fundamentals of Machine Learning for Healthcare
  - Machine Learning for All
  - Introduction to Machine Learning
  - Machine Learning with Python
  - Applied Machine Learning in Python
  - Stanford Introduction to Food and Health
  - Machine Learning for Business Professionals
  - Introduction to TensorFlow for Artificial Intelligence, Machine Learning, and Deep Learning
  - Structuring Machine Learning Projects
  - How Google does Machine Learning


## 4. Two complementary models


I now have two models: collaborative filtering (learns from ratings, strong when a user has history) and content-based (learns from course text, handles cold-start). Notebook 03 combines them into a hybrid and compares all three on precision@k / recall@k.